<a href="https://colab.research.google.com/github/CalculatedContent/WeightWatcher/blob/master/examples/XGBWW_RandomModels_XGBoost_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# XGBWW: random-model sampling + XGBoost training\n\nThis notebook:\n1. Loads the dataframe generated by `XGBWW_Dataset_Catalog_Checkpoint.ipynb`.\n2. Selects 5 random models from each source.\n3. Trains an XGBoost classifier.\n4. Reports training and test accuracy.\n

In [ ]:
# Colab install pattern: clone + setup.py (no `pip install xgbwwdata`)\nimport os, sys\n\nif 'google.colab' in sys.modules:\n    !git clone https://github.com/CalculatedContent/xgbwwdata.git\n    %cd xgbwwdata\n    !python setup.py build install\n    %cd /content\n\n!pip install -q xgboost scikit-learn pandas pyarrow\n

In [ ]:
import numpy as np\nimport pandas as pd\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split, RandomizedSearchCV\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom xgboost import XGBClassifier\n

In [ ]:
# ==== User configuration ====\nDATAFRAME_PATH = '/content/xgbww_dataset_catalog_checkpoint.parquet'\nSOURCE_COLUMN = 'source'\nTARGET_COLUMN = 'target'\nMODEL_ID_COLUMN = None  # e.g. 'model_name' if present\nRANDOM_SEED = 42\n

In [ ]:
# Load dataframe produced by XGBWW_Dataset_Catalog_Checkpoint.ipynb\npath = Path(DATAFRAME_PATH)\nassert path.exists(), f'File not found: {path}'\n\nif path.suffix.lower() in ['.parquet', '.pq']:\n    df = pd.read_parquet(path)\nelif path.suffix.lower() in ['.pkl', '.pickle']:\n    df = pd.read_pickle(path)\nelif path.suffix.lower() == '.csv':\n    df = pd.read_csv(path)\nelse:\n    raise ValueError(f'Unsupported file type: {path.suffix}')\n\nprint(f'Loaded dataframe: {df.shape[0]:,} rows x {df.shape[1]:,} columns')\ndisplay(df.head(3))\n

In [ ]:
# Select 5 random models from each source\nassert SOURCE_COLUMN in df.columns, f'Missing source column: {SOURCE_COLUMN}'\n\nsampled_df = (\n    df.groupby(SOURCE_COLUMN, group_keys=False)\n      .apply(lambda g: g.sample(n=min(5, len(g)), random_state=RANDOM_SEED))\n      .reset_index(drop=True)\n)\n\nprint('Rows after source-balanced random sampling:', sampled_df.shape[0])\ndisplay(sampled_df[[SOURCE_COLUMN]].value_counts().rename('count').reset_index(name='count'))\nif MODEL_ID_COLUMN and MODEL_ID_COLUMN in sampled_df.columns:\n    display(sampled_df[[SOURCE_COLUMN, MODEL_ID_COLUMN]].sort_values(SOURCE_COLUMN))\n

In [ ]:
# Prepare features/labels\nassert TARGET_COLUMN in sampled_df.columns, f'Missing target column: {TARGET_COLUMN}'\n\ndrop_cols = {TARGET_COLUMN, SOURCE_COLUMN}\nif MODEL_ID_COLUMN:\n    drop_cols.add(MODEL_ID_COLUMN)\n\nX = sampled_df.drop(columns=[c for c in drop_cols if c in sampled_df.columns]).copy()\nX = X.select_dtypes(include=['number', 'bool']).fillna(0)\ny = sampled_df[TARGET_COLUMN]\n\nassert X.shape[1] > 0, 'No numeric features left for training.'\nprint(f'Using {X.shape[1]} numeric feature columns')\nprint('Class distribution:')\nprint(y.value_counts(dropna=False))\n

In [ ]:
# Train/test split\nstratify = y if y.nunique() > 1 else None\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=stratify\n)\nprint('Train size:', X_train.shape, 'Test size:', X_test.shape)\n

In [ ]:
# XGBoost model + hyperparameter search (small but effective search space)\nnum_classes = y_train.nunique()\nis_binary = num_classes <= 2\n\nbase_model = XGBClassifier(\n    objective='binary:logistic' if is_binary else 'multi:softprob',\n    eval_metric='logloss' if is_binary else 'mlogloss',\n    num_class=None if is_binary else int(num_classes),\n    random_state=RANDOM_SEED,\n    tree_method='hist'\n)\n\nparam_dist = {\n    'n_estimators': [100, 200, 300, 500],\n    'max_depth': [3, 4, 5, 6, 8],\n    'learning_rate': [0.01, 0.03, 0.05, 0.1],\n    'subsample': [0.7, 0.8, 0.9, 1.0],\n    'colsample_bytree': [0.6, 0.8, 1.0],\n    'min_child_weight': [1, 3, 5],\n    'reg_alpha': [0, 0.1, 0.5],\n    'reg_lambda': [1, 2, 5],\n}\n\nsearch = RandomizedSearchCV(\n    estimator=base_model,\n    param_distributions=param_dist,\n    n_iter=20,\n    scoring='accuracy',\n    cv=3,\n    verbose=1,\n    random_state=RANDOM_SEED,\n    n_jobs=-1,\n)\nsearch.fit(X_train, y_train)\nbest_model = search.best_estimator_\nprint('Best params:', search.best_params_)\nprint('Best CV accuracy:', round(search.best_score_, 4))\n

In [ ]:
# Report training and test accuracy\ntrain_pred = best_model.predict(X_train)\ntest_pred = best_model.predict(X_test)\n\ntrain_acc = accuracy_score(y_train, train_pred)\ntest_acc = accuracy_score(y_test, test_pred)\n\nprint(f'Training accuracy: {train_acc:.4f}')\nprint(f'Test accuracy:     {test_acc:.4f}')\nprint('\nClassification report (test):')\nprint(classification_report(y_test, test_pred))\n